### sys.path Set

In [2]:
from pathlib import Path
import sys

cur = Path.cwd().resolve()
if (cur / "src").exists():
    model_dir = cur
elif (cur.parent / "src").exists():
    model_dir = cur.parent
else:
    raise FileNotFoundError("cannot find Model/src. open notebook under CVProject/Model or CVProject/Model/notebooks")

if str(model_dir) not in sys.path:
    sys.path.insert(0, str(model_dir))

print("model_dir:", model_dir)

model_dir: /root/CVProject/Model


### Import

In [3]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from pathlib import Path

from src.setup import cfg, test_dir, ckpt_dir, sub_csv_dir
from src.dataset import doc_dataset
from src.models import doc_classifier
from datetime import datetime

import re

import math
import torch.nn.functional as F

from sklearn.metrics import f1_score
import numpy as np
from pathlib import Path

print("cfg loaded ✅")
print("sample_sub path:", cfg["paths"]["sample_sub"])


/root/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cfg loaded ✅
sample_sub path: /root/CVProject/data/sample_submission.csv


### Reload

In [4]:
import importlib
import src.trainer as tr
import src.transforms as T
import pandas as pd

importlib.reload(T)
importlib.reload(tr)
print("reloaded src.transforms, src.trainer ✅")

reloaded src.transforms, src.trainer ✅


### Add TTA

In [5]:
def rot90_batch(x: torch.Tensor, k: int) -> torch.Tensor:
    """x: (B,C,H,W), k=0/1/2/3 -> 0/90/180/270 CCW"""
    if k % 4 == 0:
        return x
    return torch.rot90(x, k=k, dims=(-2, -1))

In [6]:
def rotate_affine_batch(x: torch.Tensor, angle_deg: float, mode: str = "bilinear") -> torch.Tensor:
    """
    Small-angle rotation using affine_grid + grid_sample.
    padding_mode='border' to avoid black corners (training saw white pad mostly).
    """
    if abs(angle_deg) < 1e-6:
        return x

    B, C, H, W = x.shape
    theta = angle_deg * math.pi / 180.0
    cos_t = math.cos(theta)
    sin_t = math.sin(theta)

    A = x.new_zeros((B, 2, 3))
    A[:, 0, 0] = cos_t
    A[:, 0, 1] = -sin_t
    A[:, 1, 0] = sin_t
    A[:, 1, 1] = cos_t

    grid = F.affine_grid(A, size=x.size(), align_corners=False)
    x_rot = F.grid_sample(
        x, grid,
        mode=mode,
        padding_mode="border",
        align_corners=False
    )
    return x_rot

In [7]:
@torch.no_grad()
def predict_proba_adaptive_tta(
    model,
    loader,
    use_amp: bool,
    small_angles=(-15.0, 0.0, 15.0),
):
    """
    Stage1) 0/90/180/270 각각 logits -> softmax max confidence로 best 회전 선택(샘플별)
    Stage2) best 회전 방향에서 small_angles(-15/0/+15)만 추가 적용 -> logits 평균 -> softmax
    """
    model.eval()
    probs_all, ids_all = [], []

    for x, img_id in loader:
        x = x.cuda(non_blocking=True)

        # ---- Stage 1: big rotations 0/90/180/270
        logits_list = []
        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            for k in (0, 1, 2, 3):
                xr = rot90_batch(x, k)
                logits_list.append(model(xr))  # (B, K)

        logits_stack = torch.stack(logits_list, dim=0)  # (4, B, K)
        probs_stack = torch.softmax(logits_stack, dim=-1)  # (4, B, K)
        conf = probs_stack.max(dim=-1).values  # (4, B)
        best_k = conf.argmax(dim=0)  # (B,)

        # ---- Stage 2: small angles only on chosen big rotation
        B = x.size(0)
        logits_sum = None

        for ang in small_angles:
            # big rotation apply per-sample by grouping
            x_base = x.new_empty(x.shape)
            for k in (0, 1, 2, 3):
                idx = (best_k == k).nonzero(as_tuple=True)[0]
                if idx.numel() == 0:
                    continue
                x_base[idx] = rot90_batch(x[idx], k)

            x_aug = rotate_affine_batch(x_base, float(ang))

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                logits = model(x_aug)

            logits_sum = logits if logits_sum is None else (logits_sum + logits)

        logits_avg = logits_sum / float(len(small_angles))
        p = torch.softmax(logits_avg, dim=1)

        probs_all.append(p.detach().cpu().numpy())
        ids_all.extend(list(img_id))

    probs_all = np.concatenate(probs_all, axis=0)
    return probs_all, ids_all

### Test Data Set Load + fold + submission Save

In [8]:
sample = pd.read_csv(cfg["paths"]["sample_sub"])
print(sample.head())

# 대회 샘플 제출 형식이 보통 ID, target
assert "ID" in sample.columns, "sample_submission.csv must include ID column"

test_df = sample[["ID"]].copy()
test_df.head()

                     ID  target
0  0008fdb22ddce0ce.jpg       0
1  00091bffdffd83de.jpg       0
2  00396fbc1f6cc21d.jpg       0
3  00471f8038d9c4b6.jpg       0
4  00901f504008d884.jpg       0


,ID
0,0008fdb22ddce0ce.jpg
1,00091bffdffd83de.jpg
2,00396fbc1f6cc21d.jpg
3,00471f8038d9c4b6.jpg
4,00901f504008d884.jpg


In [9]:
# sample_submission의 ID가 .jpg를 포함하지 않는 경우 자동 보정
if not str(test_df.loc[0, "ID"]).endswith(".jpg"):
    test_df["ID"] = test_df["ID"].astype(str) + ".jpg"
    print("appended .jpg to test IDs")

print("example test ID:", test_df.loc[0, "ID"])


example test ID: 0008fdb22ddce0ce.jpg


In [10]:
# ✅ infer도 train과 동일한 기준으로: 640 letterbox + bucket-valid transforms(크기 변경 없음)
base_size = cfg.get("bucket_base_size", cfg["img_size"])

test_ds = doc_dataset(
    df=test_df,
    img_dir=test_dir,
    transforms=T.get_valid_transforms_bucket(),  # ✅ 크기 조절 없는 bucket용 valid transforms
    is_test=True,
    scan_sizes=False,
    base_size=base_size,                          # ✅ 640
    use_letterbox=True                            # ✅ 비율 유지 + 패딩
)

test_loader = DataLoader(
    test_ds,
    batch_size=cfg["batch_size"],
    shuffle=False,
    num_workers=cfg["num_workers"],
    pin_memory=True
)

print("test size:", len(test_ds))


test size: 3140


In [11]:
train_df = pd.read_csv(cfg["paths"]["train_csv"])
num_classes = train_df["target"].nunique()
print("num_classes:", num_classes)

ckpt_dir = Path(ckpt_dir)


num_classes: 17


In [12]:
# ✅ 제출용: best_lb ckpt만 사용 (Dirty holdout 기준)
ckpts = list(ckpt_dir.glob("best_fold*_lb.pth"))
assert len(ckpts) == 5, f"expected 5 best_lb ckpts, found {len(ckpts)}"

def _fold_key(p: Path):
    m = re.search(r"best_fold(\d+)_lb$", p.stem)
    return int(m.group(1))

ckpts = sorted(ckpts, key=_fold_key)

available_folds = [_fold_key(p) for p in ckpts]
print("available_folds:", available_folds)  # [0,1,2,3,4] 확인용

print("found checkpoints:", [p.name for p in ckpts])

available_folds: [0, 1, 2, 3, 4]
found checkpoints: ['best_fold0_lb.pth', 'best_fold1_lb.pth', 'best_fold2_lb.pth', 'best_fold3_lb.pth', 'best_fold4_lb.pth']


In [13]:
# ✅ 제출용: uniform ensemble only
def uniform_weights(folds):
    return np.ones(len(folds), dtype=np.float32) / len(folds)

used_folds = sorted(available_folds)  # [0,1,2,3,4]
weights = uniform_weights(used_folds)

print("used_folds:", used_folds)
print("weights:", weights)

used_folds: [0, 1, 2, 3, 4]
weights: [0.2 0.2 0.2 0.2 0.2]


In [14]:
x, img_id = next(iter(test_loader))
print("[DEBUG] infer batch:", x.shape, "| example id:", img_id[0])

[DEBUG] infer batch: torch.Size([8, 3, 640, 640]) | example id: 0008fdb22ddce0ce.jpg


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = cfg["use_amp"]

probs_sum = None
ids_ref = None

for i, fold in enumerate(used_folds):
    ckpt_path = ckpt_dir / f"best_fold{fold}_lb.pth"
    print("loading:", ckpt_path.name)

    model = doc_classifier(
        model_name=cfg["model_name"],
        num_classes=num_classes,
        pretrained=False,
        dropout=cfg["dropout"]
    ).to(device)

    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)

    probs, ids = tr.predict_proba(model, test_loader, use_amp=use_amp)

    if i == 0:
        print(
            "[DEBUG probs]",
            "min:", probs.min(),
            "max:", probs.max(),
            "sum(first):", probs[0].sum()
        )

    w = float(weights[i])

    if probs_sum is None:
        probs_sum = w * probs
        ids_ref = ids
    else:
        assert list(ids) == list(ids_ref)
        probs_sum += w * probs
    
    del model, state
    torch.cuda.empty_cache()

preds = probs_sum.argmax(axis=1)

out = sample.copy()
out["target"] = preds

ver_num = datetime.now().strftime("%m%d_%H%M%S")
save_path = Path(sub_csv_dir) / f"submission_uniform_5fold_bestlb_{ver_num}.csv"
out.to_csv(save_path, index=False)

print("✅ saved:", save_path)
print("preds shape:", preds.shape)

loading: best_fold0_lb.pth


[DEBUG probs] min: 8.251599e-08 max: 0.9998672 sum(first): 1.0000001
loading: best_fold1_lb.pth


loading: best_fold2_lb.pth


 37%|███▋      | 147/393 [00:10<00:16, 14.59it/s]